# ConfigMaps und Secrets

Konfiguration wird vom Image getrennt. ConfigMaps speichern nicht vertrauliche Werte; Secrets sind für sensible Daten vorgesehen. Lernziel: Umgebungsvariablen, Startparameter und Dateien vergleichen.

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. ConfigMaps erstellen

data enthält Schlüssel und Werte. Dateien können als mehrzeilige Werte gespeichert werden.

In [ ]:
%%bash
set -eo pipefail
cat singlemap.yml fullname.yml cmfile.txt

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f singlemap.yml -f fullname.yml
kubectl create configmap cm-file-demo --from-file=cmfile.txt
kubectl get cm test-config multimap cm-file-demo -o yaml

## 2. Umgebungsvariablen und Startparameter

configMapKeyRef übernimmt einzelne Schlüssel. Kubernetes ersetzt $(FIRSTNAME) in command/args anhand der Container-Umgebung. Der Startup-Pod beendet sich nach der Ausgabe.

In [ ]:
%%bash
set -eo pipefail
cat podenv.yml podstartup.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f podenv.yml -f podstartup.yml
kubectl wait --for=condition=Ready pod/envpod --timeout=180s
kubectl exec envpod -- printenv FIRSTNAME LASTNAME
kubectl wait --for=jsonpath='{.status.phase}'=Succeeded pod/startup-pod --timeout=180s
kubectl logs startup-pod

## 3. envFrom

envFrom übernimmt alle geeigneten Schlüssel als Umgebungsvariablen. Hier entsprechen die Namen den kleingeschriebenen ConfigMap-Schlüsseln.

In [ ]:
%%bash
set -eo pipefail
cat podenvfrom.yml
kubectl apply -f podenvfrom.yml
kubectl wait --for=condition=Ready pod/envfrom-pod --timeout=180s
kubectl exec envfrom-pod -- printenv firstname lastname

## 4. ConfigMap als Volume

volumeMounts bindet das Volume in den Container ein. Jeder Schlüssel wird standardmäßig zu einer Datei.

In [ ]:
%%bash
set -eo pipefail
cat podvol.yml
kubectl apply -f podvol.yml
kubectl wait --for=condition=Ready pod/cmvol --timeout=180s
kubectl exec cmvol -- ls /etc/name
kubectl exec cmvol -- cat /etc/name/firstname

## 5. Konfiguration ändern

Umgebungsvariablen ändern sich im laufenden Container nicht. Gemountete Dateien werden verzögert aktualisiert (außer subPath). Die Schleife prüft alle fünf Sekunden, maximal etwa drei Minuten.

In [ ]:
%%bash
set -eo pipefail
kubectl patch cm multimap -p '{"data":{"firstname":"Training"}}'
kubectl exec envpod -- printenv FIRSTNAME

In [ ]:
%%bash
set -eo pipefail
for i in $(seq 1 36); do
    VALUE=$(kubectl exec cmvol -- cat /etc/name/firstname)
    echo "$VALUE"
    if [ "$VALUE" = Training ]; then break; fi
    sleep 5
done
test "$VALUE" = Training

## 6. Pod neu erzeugen

Der neue Container erhält den aktualisierten Wert Training.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f podenv.yml
kubectl apply -f podenv.yml
kubectl wait --for=condition=Ready pod/envpod --timeout=180s
kubectl exec envpod -- printenv FIRSTNAME

## 7. Secrets

Opaque ist der allgemeine Secret-Typ; weitere Typen existieren etwa für TLS und Registry-Zugangsdaten. Base64 ist keine Verschlüsselung. RBAC und Verschlüsselung at rest sind separate Schutzmaßnahmen. Nur Demo-Daten verwenden.

In [ ]:
%%bash
set -eo pipefail
cat secretpod.yml
kubectl apply -f tkb-secret.yml -f secretpod.yml
kubectl wait --for=condition=Ready pod/secret-pod --timeout=180s
kubectl get secret tkb-secret
kubectl exec secret-pod -- ls /etc/tkb

## 8. Aufräumen

Alle Beispiel-Pods und Konfigurationsobjekte entfernen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f secretpod.yml -f tkb-secret.yml -f podvol.yml -f podstartup.yml -f podenv.yml -f podenvfrom.yml -f fullname.yml -f singlemap.yml
kubectl delete cm cm-file-demo
kubectl get pods -l chapter=configmaps

## Fazit

ConfigMaps und Secrets entkoppeln Konfiguration vom Image. Ob Änderungen wirksam werden, hängt von der Art der Einbindung und der Anwendung ab.